In [ ]:
#########################################[Li2S3*+S*]#########################################
from ase.db import connect
from ase.build import add_adsorbate

db = connect('final_optimized_db.db')
# 建议写到新库，避免覆盖原始结果；如需复用旧库可改回 'Surfaces_Li2S4.db'
ads_db = connect('Surfaces_Li2S3+S_split.db')

local_xy = [(4,5), (6,6), (4,4), (6.5,4.3), (5,5), (6.5,4.3)]
z_list   = [4.2,   3.24,  3.24,  4.2,      6.0,   6.0    ]
symbols  = ['S',   'Li',  'Li',  'S',      'S',   'S'    ]

# 选择要“分解出去”的 S（当前为列表中的最后一个 S）
s_indices  = [i for i, s in enumerate(symbols) if s == 'S']
move_index = s_indices[-1]     # 想换别的 S，就改成 s_indices[0] / s_indices[1] / s_indices[2]

# 其余原子作为团簇留在右下角
stay_indices = [i for i in range(len(symbols)) if i != move_index]

# 仅用“留下的那 5 个原子”的局部坐标求几何中心 → 得到相对偏移
mx = sum(local_xy[i][0] for i in stay_indices) / len(stay_indices)
my = sum(local_xy[i][1] for i in stay_indices) / len(stay_indices)
offset_xy = {i: (local_xy[i][0] - mx, local_xy[i][1] - my) for i in stay_indices}

def frac_to_xy(atoms, fx, fy):
    # 分数坐标 → 绝对 x–y（适用于非正交晶胞）
    a, b, _ = atoms.get_cell()
    r = fx * a + fy * b
    return float(r[0]), float(r[1])

margin = 0.3  # 分数坐标边距
for row in db.select():
    atoms = row.toatoms()
    atoms_ads = atoms.copy()

    # 团簇放右下角（x→1, y→0），被拆出的 S 放左上角（x→0, y→1），都保留 margin
    brx, bry = frac_to_xy(atoms_ads, 1.0 - margin, margin)       # 右下角
    tlx, tly = frac_to_xy(atoms_ads, margin, 1.0 - margin)       # 左上角

    # 放置保留在右下角的 5 个原子（保持相对几何）
    for i in stay_indices:
        dx, dy = offset_xy[i]
        add_adsorbate(atoms_ads, symbols[i], height=z_list[i], position=(brx + dx, bry + dy))

    # 把选中的 S 单独放到左上角
    add_adsorbate(atoms_ads, symbols[move_index], height=z_list[move_index], position=(tlx, tly))

    # 防止越界缠绕
    atoms_ads.wrap(pbc=[True, True, False])

    ads_db.write(
        atoms_ads,
        model=row.formula,
        split='S_to_top_left',
        moved_S_index=int(move_index),
        margin=margin
    )


In [18]:
#########################################[Li2S2*+2S*]#########################################
from ase.db import connect 
from ase.build import add_adsorbate

db = connect('final_optimized_db.db')
# 建议写到新库避免覆盖；需要也可改回 'Surfaces_Li2S4.db'
ads_db = connect('Surfaces_Li2S2+2S.db')

# 原始平面坐标/高度/元素（Å）
local_xy = [(4,5), (6,6), (4,4), (6.5,4.3), (5,5), (6.5,4.3)]
z_list   = [4.2,   3.24,  3.24,  4.2,      6.0,   6.0    ]
symbols  = ['S',   'Li',  'Li',  'S',      'S',   'S'    ]

# 选择要“分解出去”的两个 S（默认取列表里最后两个 S）
s_indices   = [i for i, s in enumerate(symbols) if s == 'S']
move_indices = s_indices[-2:]         # 想换别的组合可改成 s_indices[0:2] 等

# 其余原子留在右下角
stay_indices = [i for i in range(len(symbols)) if i not in move_indices]

# 右下角团簇（留在表面的 4 个原子）的相对偏移（以其自身几何中心为参考）
mx_stay = sum(local_xy[i][0] for i in stay_indices) / len(stay_indices)
my_stay = sum(local_xy[i][1] for i in stay_indices) / len(stay_indices)
offset_stay = {i: (local_xy[i][0] - mx_stay, local_xy[i][1] - my_stay) for i in stay_indices}

# 左上角团簇（分解出去的 2 个 S）的相对偏移（以二者几何中心为参考）
mx_move = sum(local_xy[i][0] for i in move_indices) / len(move_indices)
my_move = sum(local_xy[i][1] for i in move_indices) / len(move_indices)
offset_move = {i: (local_xy[i][0] - mx_move, local_xy[i][1] - my_move) for i in move_indices}

def frac_to_xy(atoms, fx, fy):
    # 分数坐标 → 绝对 x–y（适用于非正交晶胞）
    a, b, _ = atoms.get_cell()
    r = fx * a + fy * b
    return float(r[0]), float(r[1])

margin = 0.3  # 分数坐标边距，避免贴边；可按需调整
for row in db.select():
    atoms = row.toatoms()
    slab = atoms.copy()

    # 目标点：右下角用于“留在一起”的 4 原子；左上角用于“分解出去”的 2 个 S
    brx, bry = frac_to_xy(slab, 1.0 - margin, margin)     # 右下
    tlx, tly = frac_to_xy(slab, margin, 1.0 - margin)     # 左上

    # 放置右下角团簇（保持相对几何）
    for i in stay_indices:
        dx, dy = offset_stay[i]
        add_adsorbate(slab, symbols[i], height=z_list[i], position=(brx + dx, bry + dy))

    # 放置左上角的两个 S（保持二者相对几何）
    for i in move_indices:
        dx, dy = offset_move[i]
        add_adsorbate(slab, symbols[i], height=z_list[i], position=(tlx + dx, tly + dy))

    # 防止越界缠绕
    slab.wrap(pbc=[True, True, False])

    ads_db.write(
        slab,
        model=row.formula,
        split='2S_to_top_left',
        margin=float(margin),                   # 标量 OK
        data={                                  # 复杂类型放 data
            'moved_S_indices': [int(i) for i in move_indices],
            'note': 'two S moved to top-left'
        }
    )

    


In [20]:
#########################################[Li2S2*+S*]#########################################
import os, shutil
from ase.db import connect
from ase.build import add_adsorbate
from ase.io import read, write

db = connect('final_optimized_db.db')
# 建议输出到新库，避免覆盖原始“右下角整体”结果；如需复用旧库可改回 'Surfaces_Li2S3.db'
ads_db = connect('Surfaces_Li2S2+S.db')

local_xy = [(6,6), (4,4), (6.5,4.3), (5,5), (6.5,4.3)]
z_list   = [3.24,  3.24,  4.2,       6.0,   6.0     ]
symbols  = ['Li',  'Li',  'S',       'S',   'S'     ]

# 选要“分解出去”的 S
s_indices  = [i for i, s in enumerate(symbols) if s == 'S']
move_index = s_indices[-1]   # 想换别的 S，就用 s_indices[0] 或 s_indices[1]

# 其余原子留在右下角
stay_indices = [i for i in range(len(symbols)) if i != move_index]

# 仅用“留下的原子”的局部坐标求几何中心 → 得到相对偏移（保持团簇内部几何）
mx = sum(local_xy[i][0] for i in stay_indices) / len(stay_indices)
my = sum(local_xy[i][1] for i in stay_indices) / len(stay_indices)
offset_xy = {i: (local_xy[i][0] - mx, local_xy[i][1] - my) for i in stay_indices}

def frac_to_xy(atoms, fx, fy):
    a, b, _ = atoms.get_cell()
    r = fx * a + fy * b
    return float(r[0]), float(r[1])

margin = 0.3  # 分数坐标边距
for row in db.select():
    slab = row.toatoms().copy()

    # 团簇放右下角，被拆出的 S 放左上角（都留 margin）
    brx, bry = frac_to_xy(slab, 1.0 - margin, margin)     # 右下
    tlx, tly = frac_to_xy(slab, margin, 1.0 - margin)     # 左上

    # 放置保留在右下角的 4 个原子（保持相对几何）
    for i in stay_indices:
        dx, dy = offset_xy[i]
        add_adsorbate(slab, symbols[i], height=z_list[i], position=(brx + dx, bry + dy))

    # 把选中的 S 单独放到左上角
    add_adsorbate(slab, symbols[move_index], height=z_list[move_index], position=(tlx, tly))

    # 防止越界缠绕
    slab.wrap(pbc=[True, True, False])

    ads_db.write(
        slab,
        model=row.formula,
        split='S_to_top_left',
        moved_S_index=int(move_index),
        margin=margin
    )

In [21]:
#########################################[Li2S*+S*]#########################################
import os, shutil
from ase.db import connect
from ase.build import add_adsorbate
from ase.io import read, write

db = connect('final_optimized_db.db')
# 建议输出到新库以便区分结果；如需覆盖原库可改回 'Surfaces_Li2S2_bottomright.db'
ads_db = connect('Surfaces_Li2S+S.db')

local_xy = [(5,5), (6,6), (4,4), (6.5,4.3)]
z_list   = [4.2,   3.24,  3.24,  3.24]
symbols  = ['S',   'Li',  'Li',  'S'   ]

# 选要“分解出去”的 S（默认最后一个 S）
s_indices  = [i for i, s in enumerate(symbols) if s == 'S']
move_index = s_indices[-1]   # 若想换另一个 S：改为 s_indices[0]

# 其余原子作为团簇留在右下角
stay_indices = [i for i in range(len(symbols)) if i != move_index]

# 仅用“留下的原子”的局部坐标求几何中心 → 相对偏移（保持团簇内部几何）
mx = sum(local_xy[i][0] for i in stay_indices) / len(stay_indices)
my = sum(local_xy[i][1] for i in stay_indices) / len(stay_indices)
offset_xy = {i: (local_xy[i][0] - mx, local_xy[i][1] - my) for i in stay_indices}

def frac_to_xy(atoms, fx, fy):
    # 分数坐标 -> 绝对 x–y（适用于非正交晶胞）
    a, b, _ = atoms.get_cell()
    r = fx * a + fy * b
    return float(r[0]), float(r[1])

margin = 0.30  # 分数坐标边距
for row in db.select():
    slab = row.toatoms().copy()

    # 团簇放右下角，被拆出的 S 放左上角（都留 margin）
    brx, bry = frac_to_xy(slab, 1.0 - margin, margin)   # 右下
    tlx, tly = frac_to_xy(slab, margin, 1.0 - margin)   # 左上

    # 放置保留在右下角的 3 个原子（保持相对几何）
    for i in stay_indices:
        dx, dy = offset_xy[i]
        add_adsorbate(slab, symbols[i], height=z_list[i], position=(brx + dx, bry + dy))

    # 把选中的 S 单独放到左上角
    add_adsorbate(slab, symbols[move_index], height=z_list[move_index], position=(tlx, tly))

    # 防止越界缠绕
    slab.wrap(pbc=[True, True, False])

    ads_db.write(
        slab,
        model=row.formula,
        split='S_to_top_left',
        moved_S_index=int(move_index),
        margin=margin
    )



In [ ]:
import os, shutil
